> ⚠️ First, you need to **select the correct environment/kernel** to run the following notebook. Please select the kernel named **"Python (Main)"** ⚠️

# Data Preprocessing & 2D-CARE Training Dataset Generation

This notebook performs the data preprocessing steps required to obtain sub-pixel paired data, and then generates the training pairs needed for the 2D-CARE training part.

The standard protocol for preprocessing is as follows:

1. Splitting the stack acquired with the beam splitter into two,
2. Sub-pixel alignment using a coefficient file **previously calculated with PALMTracer**,
3. **In the case of 2D-CARE training**, preparing the dataset for localization-informed training.

Test datasets can be download here: link. Test dataset consists of:
1. simulations of immobilized fluorophores, 
2. single particle tracking (SPT) images acquired with PALM microscopy,
3. nuclear pore complexes (NPC) acquired in DNA-PAINT. 

More details about each dataset such as acquisition modalities can be found in the dataset corresponding folder. 

## Prerequisites if running for the first time

If you are **running the notebook for the first time**, you will first need to install the packages required to smoothly run the whole pipeline. 

> If you don't have Python installed on the machine, please install it beforehand. For this repository, you will need to install a version of **Python 3.9**. We used 3.9.25 version.

To have **GPU support**, it is very important to install the specific versions of CUDA and cuDNN that are compatible with the used version of TensorFlow. In our case, we run CUDA 11.2 and cuDNN 8.0, compatible with both required versions of TensorFlow for Noise2Self (2.8.1) and CARE (2.10.1). Note that TensorFlow stopped working with Windows starting version 2.11.0.


### Windows
In the main folder, there is an *install.bat* file (Windows). Double-clicking it is enough to install the necessary virtual environments and libraries.

> You may have to download cudnn11.2, extract bin/cudnn64_8.dll and then transfer it to cuda11.2 bin for CSBDeep installation part.


### MacOS
Execute the following line one by one in a Terminal, while being in root directory:

```sh
chmod +x install.sh
./install.sh
```

> ⚠️ If you have a **libomp** error, you will need to install it with **brew**. ⚠️

Once done, **three** virtual environments should be visible: **venv**, **venv_aydin** and **venv_csbdeep**.

## Importing libraries
In the following section, we will load Python librairies required. Run the next cell to load librairies and dependencies:

In [ ]:
# Run to load libraries

import os
import sys
import tifffile

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from ipywidgets import widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output

sys.path.append(os.path.abspath('..'))

from utils.split_raw_image import split_raw_image
from utils.subpixel_transformation import subpixel_transformation


## Splitting dataset in two
An optical system containing a beam splitter is used to simultaneously generate high and low SNR images. These images are on the same sensor, so **it is necessary to split them into two**. This can be done manually (using ImageJ, for example), taking care to note the ROI size and the (0,0) coordinates of the two images relative to the original image.

The following code can load the path of one image and split it in two using:

1. Initial XY coordinates for the ROI,
2. Size of the ROI, 
3. Gap between the two images.

First, load an image:

In [ ]:
# Load the image

IMAGE_PATH = None

fc = FileChooser(os.getcwd())
fc.show_only_dirs = False
fc.filter_pattern = ['*.tif', '*.tiff', '*.stk']
fc.title = '<b>Choose raw image (.tif, .stk) :</b>'

def check_file(chooser):
    global IMAGE_PATH
    clear_output(wait=True)
    display(fc)
    
    selected_file = chooser.selected
    
    if selected_file is None:
        return
        
    if os.path.isfile(selected_file):
        IMAGE_PATH = selected_file
        print(f"Image loaded : {IMAGE_PATH}")
        print("Can now perform image splitting.")
    else:
        IMAGE_PATH = None
        print("Error : Please select an image, not a repertory or something else.")

fc.register_callback(check_file)
display(fc)

FileChooser(path='/Users/aneuhaus/Desktop/SCOL/SCOL/notebooks', filename='', title='<b>Choose raw image (.tif,…

You can know perform the splitting of the raw image loaded. **Please explore your data with ImageJ (for example) first** to enter proper values for ROI size and initial XY coordinates. 

*By default, split images are saved in the same directory as this notebook. This can be changed in the code.*

In [7]:
# Run to perform splitting
if IMAGE_PATH is None:
    print("WARNING : Please select an image first above.")
else:
    img_temp = tifffile.imread(IMAGE_PATH)
    img_display = img_temp[0] if img_temp.ndim > 2 else img_temp
    img_height_max, img_width_max = img_display.shape
    default_roi_h = (img_height_max // 2) - 15 

    w_X = widgets.IntText(value=0, description="Initial X:")
    w_Y = widgets.IntText(value=0, description="Initial Y:")
    w_W = widgets.IntText(value=img_width_max, description="Width:")
    w_H = widgets.IntText(value=default_roi_h, description="Height:")
    w_GAP = widgets.IntText(value=30, description="Gap:")
    btn_validate = widgets.Button(description="Click to split & save", button_style='success', icon='check')

    def update_preview(X, Y, W, H, GAP):
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(img_display, cmap="gray")

        rect_high = patches.Rectangle((X, Y), W, H, linewidth=2, edgecolor='red', facecolor='none', label='High SNR')
        ax.add_patch(rect_high)
        
        y2_start = Y + H + GAP
        rect_low = patches.Rectangle((X, y2_start), W, H, linewidth=2, edgecolor='dodgerblue', facecolor='none', label='Low SNR')
        ax.add_patch(rect_low)
        
        ax.legend(loc='upper right')
        ax.set_title(f"Preview: {img_width_max}x{img_height_max} px")
        plt.show()

        if y2_start + H > img_height_max or X + W > img_width_max:
            print("Error: ROI size is too large or out of bounds.")

    out_preview = widgets.interactive_output(update_preview, {'X': w_X, 'Y': w_Y, 'W': w_W, 'H': w_H, 'GAP': w_GAP})

    def on_button_clicked(b):
        X, Y, W, H, GAP = w_X.value, w_Y.value, w_W.value, w_H.value, w_GAP.value
        if (Y + H + GAP + H) > img_height_max or (X + W) > img_width_max:
            print("Error : ROIs are out boundaries !")
            return
            
        print("Saving in progress...")
        
        try:
            img_haute, img_basse = split_raw_image(IMAGE_PATH, X, Y, W, H, GAP)
            tifffile.imwrite("high_snr.tif", img_haute)
            tifffile.imwrite("low_snr.tif", img_basse)
            print("Saved !")
        except Exception as e:
            print(f"Error while saving : {e}")

    btn_validate.on_click(on_button_clicked)
    ui_controls = widgets.VBox([widgets.HBox([w_X, w_Y, w_W]), widgets.HBox([w_H, w_GAP]), btn_validate])
    display(ui_controls, out_preview)

Output()

## Subpixel Alignment of the two channels
This subsection is a necessary step, particularly for training 2D-CARE, but it can also be useful for aligning two images to compute metrics for any algorithm or comparison. The subpixel transformation is based on the following third-order polynomial function:

$$
\begin{align}
x' &= a_{x,1}x^3 + a_{x,2}y^3 + a_{x,3}x^2y + a_{x,4}xy^2 + a_{x,5}x^2 + a_{x,6}y^2 + a_{x,7}xy + a_{x,8}x + a_{x,9}y + a_{x,10} \tag{0.1} \\
y' &= a_{y,1}x^3 + a_{y,2}y^3 + a_{y,3}x^2y + a_{y,4}xy^2 + a_{y,5}x^2 + a_{y,6}y^2 + a_{y,7}xy + a_{y,8}x + a_{y,9}y + a_{y,10} \tag{0.2}
\end{align}
$$

You must change/specify the paths to the image to be calibrated and the output path (the same location as the default loaded image), as well as the 20 coefficients, which are typically contained in a text file.

<figure style="text-align: center;">
    <img src="../images/subpixel_alignment.png" width="800" />
    <figcaption style="font-style: italic; color: gray;">
        Figure 1 : Subpixel alignment calibration.
    </figcaption>
</figure>

In [ ]:
# Perform subpixel alignment

fc_txt = FileChooser(os.getcwd())
fc_txt.filter_pattern = ['*.txt']
fc_txt.title = '<b>Selectionner le fichier de coefficients (.txt) :</b>'

fc_img = FileChooser(os.getcwd())
fc_img.filter_pattern = ['*.tif', '*.stk', '*.tiff']
fc_img.title = '<b>Selectionner l\'image (.tif, .stk) :</b>'

# 2. Bouton d'exécution
btn_run = widgets.Button(
    description="Run Transformation", 
    button_style='primary', 
    icon='play'
)

# 3. Logique d'exécution
def on_run_clicked(b):
    txt_path = fc_txt.selected
    img_path = fc_img.selected
    
    if txt_path and img_path:
        # Création automatique du chemin de sortie
        base_name = os.path.splitext(img_path)[0]
        extension = os.path.splitext(img_path)[1]
        output_path = f"{base_name}_shifted{extension}"
        
        print(f"🚀 Lancement de la transformation...")
        print(f"Sortie prévue : {os.path.basename(output_path)}")
        
        try:
            # Appel de la fonction main avec le chemin de sortie calculé
            subpixel_transformation(img_path, txt_path, output_path)
            print("✅ Opération terminée avec succès !")
        except Exception as e:
            print(f"❌ Une erreur est survenue : {e}")
    else:
        print("⚠️ Veuillez sélectionner à la fois un fichier TXT et une image.")

btn_run.on_click(on_run_clicked)

# 4. Affichage
ui = widgets.VBox([fc_txt, fc_img, btn_run])
display(ui)